# Zenith on Colab

Pulls the latest `rbf22/zenith` from GitHub and runs the experiment set in the settings cell.

**One-time setup**
1. Create a fine-grained GitHub token with read access to `rbf22/zenith`.
2. In Colab, open the key icon (Secrets) and add `GITHUB_TOKEN` with that token, with notebook access on.
3. Upload data to Google Drive under `MyDrive/zenith/zenith_data/`: `train.jsonl`, `heldout.jsonl`, `tokenizer32k/`. For the resume experiment, also upload `ckpt_step9000/`.

Use a GPU runtime (Runtime > Change runtime type). T4 has no bf16 and runs fp32; L4 or A100 run bf16.

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

In [ ]:
from google.colab import drive, userdata
import base64, subprocess, os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/zenith'
DATA = f'{DRIVE}/zenith_data'
REPO = '/content/zenith'
REMOTE = 'https://github.com/rbf22/zenith.git'
auth = base64.b64encode(f'x-access-token:{userdata.get("GITHUB_TOKEN")}'.encode()).decode()
def git(*args, cwd=None):
    subprocess.run(['git', '-c', f'http.extraheader=Authorization: basic {auth}', *args], cwd=cwd, check=True)
if os.path.exists(f'{REPO}/.git'):
    git('fetch', '-q', 'origin', cwd=REPO)
    git('reset', '-q', '--hard', 'origin/main', cwd=REPO)
else:
    git('clone', '-q', REMOTE, REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], cwd=REPO, capture_output=True, text=True).stdout)

In [ ]:
!pip install -q transformers datasets tokenizers pytest pytest-cov

In [ ]:
%cd /content/zenith
!python -m pytest -q --cov=zenith --cov-report=term-missing 2>&1 | tail -15

## Settings

Choose the experiment here and re-run the cell below. Results go to Google Drive, so they survive session timeouts.

In [ ]:
EXPERIMENT = 'lr_sweep'          # 'lr_sweep' or 'main_resume'
SWEEP_LRS = ['2e-4', '1e-4']    # 300-step runs; '5e-4' already done (held-out 7.54), '2e-4' done (7.30)
MAIN_LR = '2e-4'                # peak LR for main_resume
MAIN_STEPS = 33000
RESULTS = f'{DRIVE}/results'

In [ ]:
%cd /content/zenith
import shutil, subprocess
os.makedirs(RESULTS, exist_ok=True)
def train_cmd(out, lr, steps, warmup, save, log):
    return ['python', '-m', 'zenith.train', '--config', 'configs/target/config.json', '--tokenizer', f'{DATA}/tokenizer32k',
            '--data', f'{DATA}/train.jsonl', '--output_dir', out, '--seq_len', '512', '--batch_size', '2',
            '--max_steps', str(steps), '--warmup_steps', str(warmup), '--lr', lr, '--save_steps', str(save),
            '--log_steps', str(log), '--compile_scan']
def heldout(ckpt, out):
    subprocess.run(['python', 'scripts/eval_ckpt.py', '--checkpoints', ckpt, '--tokenizer', f'{DATA}/tokenizer32k',
                    '--heldout', f'{DATA}/heldout.jsonl', '--max_windows', '128'], stdout=open(out, 'w'), stderr=subprocess.STDOUT, check=True)
    return open(out).read().strip().splitlines()[-1]
if EXPERIMENT == 'lr_sweep':
    for lr in SWEEP_LRS:
        out = f'{RESULTS}/sweep_lr{lr}'
        if os.path.exists(f'{out}/eval.log'):
            print(lr, 'already done'); continue
        os.makedirs(out, exist_ok=True)
        subprocess.run(train_cmd(f'{out}/ckpt', lr, 300, 30, 100000, 25), stdout=open(f'{out}/train.log', 'w'), stderr=subprocess.STDOUT, check=True)
        print(lr, heldout(f'{out}/ckpt', f'{out}/eval.log'))
elif EXPERIMENT == 'main_resume':
    run = f'{RESULTS}/main_lr{MAIN_LR}/ckpt'
    if not os.path.exists(run):
        shutil.copytree(f'{DRIVE}/ckpt_step9000', run)
    subprocess.run(train_cmd(run, MAIN_LR, MAIN_STEPS, 1000, 1000, 50), check=True)
else:
    raise ValueError(EXPERIMENT)

In [ ]:
%cd /content/zenith
print(heldout(f'{DRIVE}/ckpt_step9000', f'{RESULTS}/eval_step9000.log'))